# Gambar 13.1 Kata Berbobot Tertinggi Logistic Regression Akhir

*Notebook* ini membuat Gambar 13.1 untuk Subbab 13.2 laporan UTS. Gambar menampilkan 15 kata dengan bobot positif terbesar pada model Logistic Regression akhir untuk setiap kelas. Kata yang merupakan ciri korpus atau format sumber diberi warna abu-abu.

Bobot kata dibaca langsung dari output *notebook* Final Evaluation yang sudah dijalankan, sehingga model tidak perlu dilatih ulang.

In [ ]:
import json
import re
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple

ROOT = Path('/home/drew/Downloads/ML/Final Progress')
SOURCE_NOTEBOOK = ROOT / 'Final Evaluation' / 'Group_1_Email_Spam_Detection_Final_Evaluation.ipynb'
OUTPUT_FILE = ROOT / 'Report' / 'figures' / 'gambar09_kata_berpengaruh.png'

CLASS_ORDER = ['Ham', 'Phish', 'Spam']
CLASS_COLORS = {'Ham': '#2a78d6', 'Phish': '#eb6834', 'Spam': '#1baf7a'}
CORPUS_COLOR = '#8a8984'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'

plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'xtick.color': INK2, 'ytick.color': INK2,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'legend.frameon': False, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
})


def idn(x, dec=1):
    """Format angka Indonesia: 1.234,5"""
    return f'{x:,.{dec}f}'.replace(',', '#').replace('.', ',').replace('#', '.')

Sel berikut membaca tabel 15 kata teratas per kelas dari output sel interpretasi pada *notebook* Final Evaluation. Setiap baris tabel berbentuk `kata (bobot)`.

In [ ]:
notebook = json.loads(SOURCE_NOTEBOOK.read_text(encoding='utf-8'))
source_cell = next(c for c in notebook['cells']
                   if c['cell_type'] == 'code' and 'top_words[LABEL_NAMES[class_id]]' in ''.join(c['source']))
table_text = ''.join(next(o['data']['text/plain'] for o in source_cell['outputs'] if 'data' in o))

ROW = re.compile(r'^\s*\d+\s+(\S+)\s+\((-?\d+(?:\.\d+)?)\)\s*\\?$')
top_words = {cls: [] for cls in CLASS_ORDER}
current = None
for line in table_text.splitlines():
    header = line.replace('\\', '').split()
    if header and all(h in CLASS_ORDER for h in header):
        current = header[0]
        continue
    match = ROW.match(line)
    if match and current:
        top_words[current].append((match.group(1), float(match.group(2))))

for cls in CLASS_ORDER:
    print(cls, len(top_words[cls]), 'kata:', ', '.join(w if len(w) <= 24 else w[:10] + '…' for w, _ in top_words[cls]))

Kata ciri korpus atau format sumber mengikuti Subbab 13.2 laporan:

- **Ham:** staf dan lokasi Enron (`enron`, `vince`, `louise`, `houston`) serta *mailing list* perangkat lunak (`samba`, `debian`);
- **Phish:** alamat penerima korpus phishing (`josemonkeyorg`, `monkeyorg`), *header email* yang tergabung (`contenttype`, `contenttransferencoding`), nilai kosong yang tertulis sebagai teks (`nan`), dan *template newsletter* palsu (`cnncom`);
- **Spam:** universitas asal korpus (`uwaterloo`).

Kata lain diberi warna kelasnya. Daftar ini dapat diubah pada sel berikut jika kelompok menetapkan kategori lain.

In [ ]:
CORPUS_WORDS = {
    'Ham': {'enron', 'vince', 'louise', 'houston', 'samba', 'debian'},
    'Phish': {'josemonkeyorg', 'monkeyorg', 'contenttype', 'contenttransferencoding', 'nan', 'cnncom'},
    'Spam': {'uwaterloo'},
}

Sel berikut menggambar satu panel per kelas dengan sumbu bobot yang sama, lalu menyimpan gambar ke folder `Report/figures`.

In [ ]:
def short_label(word):
    if set(word) == {'_'}:
        return '_____ (garis)'
    return word if len(word) <= 24 else word[:22] + '…'


x_max = max(v for words in top_words.values() for _, v in words) * 1.18
fig, axes = plt.subplots(1, 3, figsize=(12.5, 5.6), sharex=True)
for ax, cls in zip(axes, CLASS_ORDER):
    words = top_words[cls]
    labels = [short_label(w) for w, _ in words]
    values = [v for _, v in words]
    colors = [CORPUS_COLOR if w in CORPUS_WORDS[cls] else CLASS_COLORS[cls] for w, _ in words]
    y = range(len(words))
    ax.barh(y, values, height=0.68, color=colors)
    for yi, v in zip(y, values):
        ax.text(v + x_max * 0.012, yi, idn(v), va='center', fontsize=8, color=INK)
    ax.set_yticks(list(y))
    ax.set_yticklabels(labels, fontsize=8.5, family='monospace')
    ax.invert_yaxis()
    ax.set_xlim(0, x_max)
    ax.grid(axis='y', visible=False)
    ax.set_title(cls, loc='left')
    ax.set_xlabel('Bobot koefisien')

handles = [Patch(color=CORPUS_COLOR),
           tuple(Patch(color=CLASS_COLORS[c]) for c in CLASS_ORDER)]
fig.legend(handles, ['Ciri korpus atau format sumber', 'Kata lainnya (warna kelas)'],
           handler_map={tuple: HandlerTuple(ndivide=None, pad=0.2)},
           loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.04))
fig.suptitle('15 kata berbobot tertinggi Logistic Regression akhir per kelas (teks V3)', x=0.01, ha='left')
fig.tight_layout(w_pad=2.5)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_FILE, facecolor='white')
print('Gambar disimpan:', OUTPUT_FILE, f'({OUTPUT_FILE.stat().st_size:,} byte)')
plt.show()